# 1. Project Overview

**Project Title:** Email Spam Detection Using Natural Language Processing (NLP)  
**Objective:** Develop a machine learning system that accurately classifies incoming emails into two binary categories:  
- **Spam (1):** Unsolicited, commercial, or fraudulent emails.  
- **Not Spam (0):** Legitimate personal, organizational, or transactional emails.  

This project implements an end-to-end, reproducible ML/NLP pipeline using the **SpamAssassin Public Corpus (2002/2003)**. This notebook contains Stage 3B.1: Dataset Overview and Basic Exploratory Data Analysis (EDA).

# 2. Problem Statement

Given the raw text content, headers, and metadata of an incoming email message, the objective is to build a binary classification system that reliably determines whether the message is legitimate (`Not Spam` / `Ham`) or illegitimate (`Spam`).

A central challenge in email spam detection is the **asymmetric cost of misclassification**:
- **False Negative (Spam classified as Not Spam):** Minor nuisance or potential security hazard delivered to the inbox.
- **False Positive (Legitimate email classified as Spam):** Critical failure causing important communications (job offers, academic notices, financial invoices) to be lost or quarantined.

Thus, an effective spam filter must maintain an exceptionally low False Positive Rate (FPR) while maximizing Spam Precision and Recall.

# 3. Dataset Description

This project utilizes the **SpamAssassin Public Corpus (20030228 release)**.
The source dataset contains three original categories:
1. `easy_ham`: Legitimate emails with typical personal/work linguistic structure (2,472 cleaned emails).
2. `hard_ham`: Legitimate emails with HTML formatting or technical mailing list headers that closely resemble spam (248 cleaned emails).
3. `spam`: Unsolicited commercial or fraudulent emails (481 cleaned emails).

Binary label mapping:
- `0 = Not Spam` (`easy_ham`, `hard_ham`)
- `1 = Spam` (`spam`)

The cleaned, deduplicated dataset is loaded from `dataset/spam_assassin_cleaned.csv`.

# 4. Import Libraries

We import required libraries for data manipulation, numerical analysis, visualization, and path management.

In [ ]:
import os
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization styling
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "Arial"
plt.rcParams["axes.edgecolor"] = "#333333"
plt.rcParams["axes.linewidth"] = 0.8

print("Libraries successfully imported.")

# 5. Load Dataset

We resolve project paths dynamically and load the verified cleaned dataset (`dataset/spam_assassin_cleaned.csv`).

In [ ]:
# Dynamically resolve project root
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "dataset").exists() else Path.cwd().parent
DATASET_PATH = PROJECT_ROOT / "dataset" / "spam_assassin_cleaned.csv"
GRAPHS_DIR = PROJECT_ROOT / "results" / "graphs"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

# Load cleaned dataset
df = pd.read_csv(DATASET_PATH, keep_default_na=False)
print(f"Loaded dataset from: {DATASET_PATH}")
print(f"Dataset Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")

# 6. Initial Dataset Inspection

Inspect the first five records, schema, column data types, and memory utilization.

In [ ]:
# Display first five rows
df.head()

In [ ]:
# Display dataset shape and schema summary
print("--- Dataset Information ---")
df.info()

print("\n--- Memory Usage per Column ---")
mem = df.memory_usage(deep=True)
for col, bytes_used in mem.items():
    print(f"{col:<15}: {bytes_used / (1024 * 1024):.2f} MB")
print(f"Total Memory   : {mem.sum() / (1024 * 1024):.2f} MB")

# 7. Data Quality Checks

We verify that missing values, empty strings, whitespace-only records, invalid labels, and text duplicates are absent.

In [ ]:
# 1. Missing values check
missing_counts = df.isna().sum()

# 2. Empty and whitespace-only text checks
empty_text = (df["text"] == "").sum()
ws_text = (df["text"].str.strip() == "").sum()
empty_subj = (df["subject"] == "").sum()
ws_subj = (df["subject"].str.strip() == "").sum()

# 3. Label validity
unique_labels = sorted(df["label"].unique().tolist())
invalid_labels = (~df["label"].isin([0, 1])).sum()

# 4. Text duplicates
text_duplicates = df["text"].duplicated().sum()

quality_summary = pd.DataFrame([
    {"Check": "Missing Values (Any Column)", "Result": missing_counts.sum()},
    {"Check": "Empty text Strings", "Result": empty_text},
    {"Check": "Whitespace-Only text Records", "Result": ws_text},
    {"Check": "Empty / Whitespace-Only Subjects", "Result": f"{empty_subj} (legitimate emails with omitted subject)"},
    {"Check": "Unique Target Labels", "Result": str(unique_labels)},
    {"Check": "Invalid Target Labels (not in {0, 1})", "Result": invalid_labels},
    {"Check": "Duplicate Full-Text Records", "Result": text_duplicates}
])
quality_summary

# 8. Exploratory Data Analysis

We conduct statistical and visual exploration of class distributions, category breakdowns, and document length distributions.

### 8.1 Target Class Distribution

In [ ]:
# Calculate class distribution
class_counts = df["label"].value_counts().rename({0: "Not Spam (0)", 1: "Spam (1)"})
class_pcts = (class_counts / len(df) * 100).round(2)
class_df = pd.DataFrame({"Count": class_counts, "Percentage (%)": class_pcts})
display(class_df)

# Plot and save class distribution bar chart
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(class_counts.index, class_counts.values, color=["#2b5c8f", "#d95f02"], width=0.5, edgecolor="black", linewidth=0.8)
for bar in bars:
    yval = bar.get_height()
    pct = yval / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 40, f"{yval:,}\n({pct:.2f}%)", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("Target Class Distribution in Cleaned SpamAssassin Dataset", fontsize=13, fontweight="bold", pad=15)
ax.set_ylabel("Number of Emails", fontsize=11)
ax.set_ylim(0, 3100)
plt.tight_layout()
class_plot_path = GRAPHS_DIR / "class_distribution.png"
plt.savefig(class_plot_path, dpi=300)
plt.show()
print(f"Saved: {class_plot_path}")

### 8.2 Original Category Distribution

The dataset originates from three SpamAssassin corpus tarballs: `easy_ham` and `hard_ham` both map to **Not Spam (0)**, while `spam` maps to **Spam (1)**. Retaining `hard_ham` provides a crucial benchmark for identifying false positives on challenging legitimate emails.

In [ ]:
# Calculate category distribution
cat_counts = df["category"].value_counts()
cat_order = ["easy_ham", "hard_ham", "spam"]
cat_values = [cat_counts[c] for c in cat_order]
cat_pcts = [(cat_counts[c] / len(df) * 100).round(2) for c in cat_order]
cat_labels = ["easy_ham (Not Spam - 0)", "hard_ham (Not Spam - 0)", "spam (Spam - 1)"]
cat_df = pd.DataFrame({"Category": cat_order, "Mapped Class": ["Not Spam (0)", "Not Spam (0)", "Spam (1)"], "Count": cat_values, "Percentage (%)": cat_pcts})
display(cat_df)

# Plot and save category distribution
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(cat_labels, cat_values, color=["#2b5c8f", "#41b6c4", "#d95f02"], width=0.55, edgecolor="black", linewidth=0.8)
for bar in bars:
    yval = bar.get_height()
    pct = yval / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 35, f"{yval:,}\n({pct:.2f}%)", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("Original Category Distribution (SpamAssassin 20030228)", fontsize=13, fontweight="bold", pad=15)
ax.set_ylabel("Number of Emails", fontsize=11)
ax.set_ylim(0, 2800)
plt.tight_layout()
cat_plot_path = GRAPHS_DIR / "category_distribution.png"
plt.savefig(cat_plot_path, dpi=300)
plt.show()
print(f"Saved: {cat_plot_path}")

### 8.3 Email and Subject Length Analysis

We compute temporary length and word count metrics in notebook memory without altering the underlying dataset CSV:
- `character_count`: total characters in combined email text.
- `word_count`: total whitespace-separated words in email text.
- `subject_character_count`: characters in the subject header.
- `subject_word_count`: words in the subject header.

In [ ]:
# Create temporary analysis features in memory
df_analysis = df.copy()
df_analysis["character_count"] = df_analysis["text"].str.len()
df_analysis["word_count"] = df_analysis["text"].apply(lambda x: len(x.split()))
df_analysis["subject_character_count"] = df_analysis["subject"].str.len()
df_analysis["subject_word_count"] = df_analysis["subject"].apply(lambda x: len(x.split()))

feature_cols = ["character_count", "word_count", "subject_character_count", "subject_word_count"]

# Descriptive statistics across entire dataset
def compute_descriptive_stats(series):
    return {
        "Count": int(series.count()),
        "Mean": round(series.mean(), 2),
        "Median": round(series.median(), 2),
        "Std Dev": round(series.std(), 2),
        "Min": int(series.min()),
        "25%": round(series.quantile(0.25), 2),
        "75%": round(series.quantile(0.75), 2),
        "Max": int(series.max())
    }

overall_stats_df = pd.DataFrame([compute_descriptive_stats(df_analysis[c]) for c in feature_cols], index=feature_cols)
print("Overall Descriptive Statistics:")
display(overall_stats_df)

In [ ]:
# Grouped statistics by target class
class_comparison_stats = {}
for col in feature_cols:
    grp = df_analysis.groupby("label")[col].agg([
        "count", "mean", "median", "std", "min",
        lambda x: x.quantile(0.25),
        lambda x: x.quantile(0.75),
        "max"
    ]).rename(columns={"<lambda_0>": "25%", "<lambda_1>": "75%"}).round(2)
    grp.index = ["Not Spam (0)", "Spam (1)"]
    class_comparison_stats[col] = grp

print("--- Character Count by Class ---")
display(class_comparison_stats["character_count"])

print("--- Word Count by Class ---")
display(class_comparison_stats["word_count"])

print("--- Subject Character Count by Class ---")
display(class_comparison_stats["subject_character_count"])

print("--- Subject Word Count by Class ---")
display(class_comparison_stats["subject_word_count"])

In [ ]:
# Plot and save Email Character Count Distribution (Log scale + Boxplot)
fig, (ax_box, ax_hist) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw={"height_ratios": [0.3, 0.7]})
sns.boxplot(data=df_analysis, x="character_count", y="label", hue="label", orient="h", ax=ax_box, palette=["#2b5c8f", "#d95f02"], legend=False, showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
ax_box.set_yticks([0, 1])
ax_box.set_yticklabels(["Not Spam (0)", "Spam (1)"])
ax_box.set_xlabel("")
ax_box.set_title("Email Character Count Distribution by Target Class", fontsize=13, fontweight="bold", pad=10)

bins = np.logspace(np.log10(max(10, df_analysis["character_count"].min())), np.log10(df_analysis["character_count"].max()), 50)
ax_hist.hist(df_analysis[df_analysis["label"]==0]["character_count"], bins=bins, alpha=0.6, label="Not Spam (0)", color="#2b5c8f", edgecolor="black", linewidth=0.5)
ax_hist.hist(df_analysis[df_analysis["label"]==1]["character_count"], bins=bins, alpha=0.6, label="Spam (1)", color="#d95f02", edgecolor="black", linewidth=0.5)
ax_hist.set_xscale("log")
ax_hist.set_xlabel("Character Count (Log Scale)", fontsize=11)
ax_hist.set_ylabel("Frequency", fontsize=11)
ax_hist.legend(title="Target Class", frameon=True)
plt.tight_layout()
email_len_plot = GRAPHS_DIR / "email_length_distribution.png"
plt.savefig(email_len_plot, dpi=300)
plt.show()
print(f"Saved: {email_len_plot}")

In [ ]:
# Plot and save Email Word Count Distribution (Log scale + Boxplot)
fig, (ax_box, ax_hist) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw={"height_ratios": [0.3, 0.7]})
sns.boxplot(data=df_analysis, x="word_count", y="label", hue="label", orient="h", ax=ax_box, palette=["#2b5c8f", "#d95f02"], legend=False, showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
ax_box.set_yticks([0, 1])
ax_box.set_yticklabels(["Not Spam (0)", "Spam (1)"])
ax_box.set_xlabel("")
ax_box.set_title("Email Word Count Distribution by Target Class", fontsize=13, fontweight="bold", pad=10)

bins_words = np.logspace(np.log10(max(1, df_analysis["word_count"].min())), np.log10(df_analysis["word_count"].max()), 50)
ax_hist.hist(df_analysis[df_analysis["label"]==0]["word_count"], bins=bins_words, alpha=0.6, label="Not Spam (0)", color="#2b5c8f", edgecolor="black", linewidth=0.5)
ax_hist.hist(df_analysis[df_analysis["label"]==1]["word_count"], bins=bins_words, alpha=0.6, label="Spam (1)", color="#d95f02", edgecolor="black", linewidth=0.5)
ax_hist.set_xscale("log")
ax_hist.set_xlabel("Word Count (Log Scale)", fontsize=11)
ax_hist.set_ylabel("Frequency", fontsize=11)
ax_hist.legend(title="Target Class", frameon=True)
plt.tight_layout()
word_plot = GRAPHS_DIR / "word_count_distribution.png"
plt.savefig(word_plot, dpi=300)
plt.show()
print(f"Saved: {word_plot}")

In [ ]:
# Plot and save Subject Character and Word Count Distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=df_analysis, x="label", y="subject_character_count", hue="label", ax=ax1, palette=["#2b5c8f", "#d95f02"], legend=False, showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
ax1.set_xticks([0, 1])
ax1.set_xticklabels(["Not Spam (0)", "Spam (1)"])
ax1.set_xlabel("Target Class", fontsize=11)
ax1.set_ylabel("Subject Character Count", fontsize=11)
ax1.set_title("Subject Character Count by Class", fontsize=12, fontweight="bold")

sns.boxplot(data=df_analysis, x="label", y="subject_word_count", hue="label", ax=ax2, palette=["#2b5c8f", "#d95f02"], legend=False, showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
ax2.set_xticks([0, 1])
ax2.set_xticklabels(["Not Spam (0)", "Spam (1)"])
ax2.set_xlabel("Target Class", fontsize=11)
ax2.set_ylabel("Subject Word Count", fontsize=11)
ax2.set_title("Subject Word Count by Class", fontsize=12, fontweight="bold")
plt.suptitle("Email Subject Length Comparisons by Target Class", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
subj_plot = GRAPHS_DIR / "subject_length_distribution.png"
plt.savefig(subj_plot, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {subj_plot}")

### 8.4 Initial Observations from Exploratory Data Analysis

Based strictly on the computed empirical results:
1. **Moderate Class Imbalance:**
   - Not Spam (`0`) accounts for **84.97%** (2,720 emails), while Spam (`1`) accounts for **15.03%** (481 emails), representing an approximate 5.65:1 imbalance ratio.
   - This confirms that Accuracy alone will be misleading (a naive majority-class model achieves ~85% accuracy), necessitating precision, recall, F1-score, and PR-AUC optimization.
2. **Category Breakdown:**
   - Legitimate emails are subdivided into `easy_ham` (2,472 emails, 77.23%) and `hard_ham` (248 emails, 7.75%). `hard_ham` provides a vital set of false-positive stress test cases.
3. **Email Length Differences:**
   - **Spam emails exhibit higher central tendencies:** Mean character count for Spam is **2,323.83** vs. **1,899.86** for Ham; median character count is **1,134.0** for Spam vs. **963.0** for Ham.
   - Mean word count is **325.18** for Spam vs. **279.77** for Ham; median word count is **160.0** for Spam vs. **140.0** for Ham.
4. **Subject Length Differences:**
   - Spam subject lines are moderately longer on average: **39.95 characters** (mean) / **38.0 characters** (median) for Spam vs. **37.20 characters** (mean) / **34.0 characters** (median) for Ham.
   - Subject word counts show a median of **6.0 words** for Spam vs. **5.0 words** for Ham.
5. **Distribution Skewness and Extreme Outliers:**
   - Both classes exhibit severe right-skewed length distributions. Maximum email character count reaches **194,850 characters** (14,959 words) in Ham (large technical digest/log emails) and **126,362 characters** (11,455 words) in Spam.
   - Minimum character count is 59 in Ham and 11 in Spam.
   - Exactly **3 emails** in Ham have empty subject lines (`""`), corresponding to legitimate RFC-822 messages that were transmitted without a `Subject:` header.

### 8.5 Class-Independent Vocabulary & N-Gram Analysis

> **Corpus Limitation Caveat:** The text extracted from raw SpamAssassin messages contains residual RFC-822 headers, server routing lines, and mailing-list metadata (e.g., `http`, `com`, `zdnet`, `rpm list`, `mailman listinfo`) that naturally influence raw token frequencies. Vocabulary profiling reflects these authentic corpus characteristics.

To extract true class-specific terms without cross-contamination, we fit two independent Scikit-learn `CountVectorizer` instances (with identical parameters: `stop_words='english'`, `lowercase=True`, `binary=False`):
- **Vectorizer A:** Fit strictly on Not Spam (0) emails ($N = 2,720$).
- **Vectorizer B:** Fit strictly on Spam (1) emails ($N = 481$).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

ham_text = df["text"][df["label"] == 0]
spam_text = df["text"][df["label"] == 1]

print(f"Not Spam (0) sample size: {len(ham_text):,} emails")
print(f"Spam (1) sample size:     {len(spam_text):,} emails")

# 1. Unigrams
vec_u_ham = CountVectorizer(stop_words="english", lowercase=True, ngram_range=(1, 1))
X_u_ham = vec_u_ham.fit_transform(ham_text)
top20_u_ham = pd.DataFrame({
    "Term": vec_u_ham.get_feature_names_out(),
    "Count": np.asarray(X_u_ham.sum(axis=0)).flatten()
}).sort_values("Count", ascending=False).head(20).reset_index(drop=True)

vec_u_spam = CountVectorizer(stop_words="english", lowercase=True, ngram_range=(1, 1))
X_u_spam = vec_u_spam.fit_transform(spam_text)
top20_u_spam = pd.DataFrame({
    "Term": vec_u_spam.get_feature_names_out(),
    "Count": np.asarray(X_u_spam.sum(axis=0)).flatten()
}).sort_values("Count", ascending=False).head(20).reset_index(drop=True)

# 2. Bigrams
vec_b_ham = CountVectorizer(stop_words="english", lowercase=True, ngram_range=(2, 2))
X_b_ham = vec_b_ham.fit_transform(ham_text)
top15_b_ham = pd.DataFrame({
    "Bigram": vec_b_ham.get_feature_names_out(),
    "Count": np.asarray(X_b_ham.sum(axis=0)).flatten()
}).sort_values("Count", ascending=False).head(15).reset_index(drop=True)

vec_b_spam = CountVectorizer(stop_words="english", lowercase=True, ngram_range=(2, 2))
X_b_spam = vec_b_spam.fit_transform(spam_text)
top15_b_spam = pd.DataFrame({
    "Bigram": vec_b_spam.get_feature_names_out(),
    "Count": np.asarray(X_b_spam.sum(axis=0)).flatten()
}).sort_values("Count", ascending=False).head(15).reset_index(drop=True)

# 3. Trigrams
vec_t_ham = CountVectorizer(stop_words="english", lowercase=True, ngram_range=(3, 3))
X_t_ham = vec_t_ham.fit_transform(ham_text)
top10_t_ham = pd.DataFrame({
    "Trigram": vec_t_ham.get_feature_names_out(),
    "Count": np.asarray(X_t_ham.sum(axis=0)).flatten()
}).sort_values("Count", ascending=False).head(10).reset_index(drop=True)

vec_t_spam = CountVectorizer(stop_words="english", lowercase=True, ngram_range=(3, 3))
X_t_spam = vec_t_spam.fit_transform(spam_text)
top10_t_spam = pd.DataFrame({
    "Trigram": vec_t_spam.get_feature_names_out(),
    "Count": np.asarray(X_t_spam.sum(axis=0)).flatten()
}).sort_values("Count", ascending=False).head(10).reset_index(drop=True)

print("=== Top 10 Trigrams Comparison ===")
trigram_comp = pd.concat([top10_t_ham.rename(columns=lambda c: f"Not Spam (0) {c}"), top10_t_spam.rename(columns=lambda c: f"Spam (1) {c}")], axis=1)
display(trigram_comp)

In [ ]:
# Plot and save Top 20 Unigrams Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 8))
u_ham_plot = top20_u_ham.sort_values("Count", ascending=True)
u_spam_plot = top20_u_spam.sort_values("Count", ascending=True)

ax1.barh(u_ham_plot["Term"], u_ham_plot["Count"], color="#2b5c8f", edgecolor="black", linewidth=0.6)
ax1.set_title("Top 20 Unigrams - Not Spam (0) (N=2,720)", fontsize=12, fontweight="bold", pad=10)
ax1.set_xlabel("Term Frequency (Raw Count)", fontsize=11)

ax2.barh(u_spam_plot["Term"], u_spam_plot["Count"], color="#d95f02", edgecolor="black", linewidth=0.6)
ax2.set_title("Top 20 Unigrams - Spam (1) (N=481)", fontsize=12, fontweight="bold", pad=10)
ax2.set_xlabel("Term Frequency (Raw Count)", fontsize=11)

plt.suptitle("Class-Independent Top 20 Unigrams Comparison (Stop Words = English)", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
unigram_path = GRAPHS_DIR / "top_unigrams_comparison.png"
plt.savefig(unigram_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {unigram_path}")

In [ ]:
# Plot and save Top 15 Bigrams Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
b_ham_plot = top15_b_ham.sort_values("Count", ascending=True)
b_spam_plot = top15_b_spam.sort_values("Count", ascending=True)

ax1.barh(b_ham_plot["Bigram"], b_ham_plot["Count"], color="#2b5c8f", edgecolor="black", linewidth=0.6)
ax1.set_title("Top 15 Bigrams - Not Spam (0) (N=2,720)", fontsize=12, fontweight="bold", pad=10)
ax1.set_xlabel("Term Frequency (Raw Count)", fontsize=11)

ax2.barh(b_spam_plot["Bigram"], b_spam_plot["Count"], color="#d95f02", edgecolor="black", linewidth=0.6)
ax2.set_title("Top 15 Bigrams - Spam (1) (N=481)", fontsize=12, fontweight="bold", pad=10)
ax2.set_xlabel("Term Frequency (Raw Count)", fontsize=11)

plt.suptitle("Class-Independent Top 15 Bigrams Comparison (Stop Words = English)", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
bigram_path = GRAPHS_DIR / "top_bigrams_comparison.png"
plt.savefig(bigram_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {bigram_path}")

### 8.6 Structural Heuristics & Pattern Profiling

In addition to vocabulary tokens, spam detection commonly leverages structural and formatting signals. In notebook memory only, we extract six standardized heuristic features:
1. `url_count`: Matches for `re.findall(r"https?://\S+|www\.\S+", text)` (identifies explicit links).
2. `exclamation_count`: Total occurrences of `!` in text.
3. `question_count`: Total occurrences of `?` in text.
4. `currency_symbol_count`: Matches for `re.findall(r"[$€£¥]", text)` (monetary symbols).
5. `uppercase_ratio`: Ratio of uppercase alphabetic characters to total alphabetic characters (`isupper() / max(isalpha(), 1)`).
6. `all_caps_word_count`: Words with $\ge 3$ characters composed exclusively of uppercase letters (`\b[A-Z]{3,}\b`).

In [ ]:
import re

# Compute 6 heuristic features in memory
df_heuristics = pd.DataFrame(index=df.index)
df_heuristics["label"] = df["label"]

df_heuristics["url_count"] = df["text"].apply(lambda t: len(re.findall(r"https?://\S+|www\.\S+", t)))
df_heuristics["exclamation_count"] = df["text"].apply(lambda t: t.count("!"))
df_heuristics["question_count"] = df["text"].apply(lambda t: t.count("?"))
df_heuristics["currency_symbol_count"] = df["text"].apply(lambda t: len(re.findall(r"[$€£¥]", t)))
df_heuristics["uppercase_ratio"] = df["text"].apply(
    lambda t: sum(1 for c in t if c.isupper()) / max(sum(1 for c in t if c.isalpha()), 1)
)
df_heuristics["all_caps_word_count"] = df["text"].apply(lambda t: len(re.findall(r"\b[A-Z]{3,}\b", t)))

heuristic_cols = ["url_count", "exclamation_count", "question_count", "currency_symbol_count", "uppercase_ratio", "all_caps_word_count"]

# Grouped descriptive statistics table
heuristic_summary_list = []
for col in heuristic_cols:
    grp = df_heuristics.groupby("label")[col].agg([
        "count", "mean", "median", "std", "min",
        lambda x: x.quantile(0.25),
        lambda x: x.quantile(0.75),
        "max"
    ]).rename(columns={"<lambda_0>": "25%", "<lambda_1>": "75%"}).round(4)
    grp.index = ["Not Spam (0)", "Spam (1)"]
    grp["feature"] = col
    heuristic_summary_list.append(grp.reset_index().rename(columns={"index": "Class"}))

heuristic_summary_df = pd.concat(heuristic_summary_list).set_index(["feature", "Class"])
print("=== Structural Heuristics Comparison (Grouped by Class) ===")
display(heuristic_summary_df)

In [ ]:
# Plot and save Heuristic Comparison Multi-Panel Figure (2x2 Grid)
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. URL Count (Log Scale Boxplot)
sns.boxplot(data=df_heuristics, x="label", y="url_count", hue="label", ax=axes[0, 0], palette=["#2b5c8f", "#d95f02"], legend=False, showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
axes[0, 0].set_yscale("log")
axes[0, 0].set_xticks([0, 1])
axes[0, 0].set_xticklabels(["Not Spam (0)", "Spam (1)"])
axes[0, 0].set_title("URL Count by Class (Log Scale)", fontsize=12, fontweight="bold")
axes[0, 0].set_xlabel("Target Class", fontsize=11)
axes[0, 0].set_ylabel("URLs per Email (Log Scale)", fontsize=11)

# 2. Exclamation Count (Log Scale Boxplot)
sns.boxplot(data=df_heuristics, x="label", y="exclamation_count", hue="label", ax=axes[0, 1], palette=["#2b5c8f", "#d95f02"], legend=False, showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
axes[0, 1].set_yscale("log")
axes[0, 1].set_xticks([0, 1])
axes[0, 1].set_xticklabels(["Not Spam (0)", "Spam (1)"])
axes[0, 1].set_title("Exclamation Mark (!) Count by Class (Log Scale)", fontsize=12, fontweight="bold")
axes[0, 1].set_xlabel("Target Class", fontsize=11)
axes[0, 1].set_ylabel("Exclamation Count (Log Scale)", fontsize=11)

# 3. Currency Count (Log Scale Boxplot)
sns.boxplot(data=df_heuristics, x="label", y="currency_symbol_count", hue="label", ax=axes[1, 0], palette=["#2b5c8f", "#d95f02"], legend=False, showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
axes[1, 0].set_yscale("log")
axes[1, 0].set_xticks([0, 1])
axes[1, 0].set_xticklabels(["Not Spam (0)", "Spam (1)"])
axes[1, 0].set_title("Currency Symbols ($, €, £, ¥) Count (Log Scale)", fontsize=12, fontweight="bold")
axes[1, 0].set_xlabel("Target Class", fontsize=11)
axes[1, 0].set_ylabel("Currency Count (Log Scale)", fontsize=11)

# 4. Uppercase Ratio (Linear Boxplot)
sns.boxplot(data=df_heuristics, x="label", y="uppercase_ratio", hue="label", ax=axes[1, 1], palette=["#2b5c8f", "#d95f02"], legend=False, showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
axes[1, 1].set_xticks([0, 1])
axes[1, 1].set_xticklabels(["Not Spam (0)", "Spam (1)"])
axes[1, 1].set_title("Uppercase Alphabetic Ratio by Class", fontsize=12, fontweight="bold")
axes[1, 1].set_xlabel("Target Class", fontsize=11)
axes[1, 1].set_ylabel("Uppercase Characters / Total Letters", fontsize=11)

plt.suptitle("Structural Heuristics Comparison: Not Spam (0) vs. Spam (1)", fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
heuristic_fig_path = GRAPHS_DIR / "url_punctuation_heuristics.png"
plt.savefig(heuristic_fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {heuristic_fig_path}")

### 8.7 Empirical Observations & Feature Engineering Rationale

#### Vectorizer & Heuristic Configuration

All N-gram counts in this section are produced by **Cell 25** using two class-independent
`CountVectorizer` instances fit separately on Not Spam (0) and Spam (1) emails:

| Parameter | Value |
|:---|:---|
| **Source field** | `df["text"]` (full raw email: RFC-822 headers + body) |
| **`stop_words`** | `"english"` (scikit-learn built-in list) |
| **`lowercase`** | `True` |
| **`token_pattern`** | Default `(?u)\b\w\w+\b` (2+ character alphanumeric tokens) |
| **`binary`** | `False` (raw term frequency) |
| **Dataset** | `spam_assassin_cleaned.csv` (N = 3,201; Not Spam N = 2,720; Spam N = 481) |

URL heuristics (Cell 29) use regex `r'https?://\S+|www\.\S+'`.

---

#### Directly Measured Findings:
1. **Vocabulary Divergence:**
   - **Not Spam (0):** Top unigrams are dominated by web infrastructure tokens and open-source
     mailing-list artefacts: `http` (8,709), `com` (7,923), `www` (4,915), `net` (2,976),
     `2002` (2,673), `list` (2,328), `use` (1,643), `org` (1,618), `new` (1,614), `html` (1,605).
     Key bigrams include `http www` (4,605), `rpm list` (778), `mailing list` (750),
     `freshrpms net` (721), and `sourceforge net` (655).
     Top trigrams: `lists sourceforge net` (564), `url http www` (393),
     `radio weblogs com` (371), `http radio weblogs` (371), `www newsisfree com` (352).
   - **Spam (1):** Top unigrams reflect commercial and monetary propositions:
     `com` (745), `free` (734), `http` (684), `000` (531), `email` (519),
     `money` (510), `www` (497), `mail` (431), `click` (386), `list` (383).
     Key bigrams include `http www` (443), `email address` (78), `united states` (77),
     `wish receive` (73), and `000 000` (73).
     Top trigrams: `www insuranceiq com` (63), `http www insuranceiq` (63),
     `http www adclick` (56), `www adclick ws` (56), `adclick ws cfm` (56).
   - **Shared Web Syntax:** Generic URL tokens (`http`, `com`, `www`, `net`) rank highly in
     **both** classes. In Not Spam they arise from mailing-list footers, patch-tracker URLs,
     and aggregator digests; in Spam they arise from click-through link redirectors. These
     tokens are corpus artefacts of the 2002/2003 SpamAssassin collection and should be
     interpreted with the Corpus Limitation Caveat from Section 8.5 in mind.

2. **URL Heuristics** (regex `r'https?://\S+|www\.\S+'`):
   - **Not Spam (0):** 2,161 of 2,720 emails **(79.45%)** contain at least one URL;
     mean URL count = **3.42**; median = 1; maximum = **3,133**
     (Row 2698, `hard_ham`, subject: *“updated weblogs from blo.gs”* — a weblog aggregation
     directory digest).
   - **Spam (1):** 291 of 481 emails **(60.50%)** contain at least one URL;
     mean URL count = **1.50**; median = 1; maximum = 13.
   - Counterintuitively, Not Spam exhibits a higher mean URL count than Spam. The extreme
     right-tail outlier (3,133) is driven by a single open-source aggregator digest.
     Spam emails in this corpus typically employ 1–2 targeted click-through links.

3. **Heuristic Discrepancies:**
   - **Exclamation Marks (`!`):** Mean count is nearly $5\times$ higher in Spam (**4.95** vs.
     **1.02**); Spam 75th percentile = **6.0** vs. **1.0** for Not Spam.
   - **Currency Symbols (`$`, `€`, `£`, `¥`):** Mean count is over $5\times$ higher in
     Spam (**3.42** vs. **0.63**); 75th percentile = **2.0** in Spam vs. **0.0** in Not Spam.
   - **Uppercase Casing Ratio:** Average uppercase ratio is approximately double in Spam
     (**12.81%** vs. **6.58%**); median **9.67%** vs. **5.83%**.
   - **ALL-CAPS Words ($\ge 3$ letters):** Mean in Spam = **18.83** (median **7.0**,
     75th pct **17.0**) vs. Not Spam mean **5.42** (median **2.0**, 75th pct **4.0**).

#### Corpus Artefacts & Nuances:
- High-frequency tokens such as `http`, `com`, `freshrpms`, `sourceforge`, and `listinfo`
  represent mailing-list headers and digests characteristic of the 2002/2003 SpamAssassin
  collection. They are **not** universal indicators of legitimate email and will be handled
  during Stage 4 feature engineering.
- These empirical differences describe corpus-level tendencies; no single heuristic or token
  is individually sufficient or determinative for classification.

#### Academic Rationale for Preprocessing Design (Stage 4):
- Standard aggressive preprocessing (lowercasing, punctuation stripping) discards currency
  symbols, exclamation marks, and casing signals that carry strong discriminating power in
  this corpus.
- In Stage 4, the Scikit-learn Pipeline should preserve — or explicitly engineer — casing
  intensity ratios, punctuation density, and currency frequency alongside TF-IDF unigrams
  and bigrams.


# 9. Text Preprocessing

*(Planned for Stage 4: Text cleaning, tokenization, stop-word handling, and Scikit-learn Pipeline feature extraction without data leakage.)*

# 10. Baseline Model

*(Planned for Stage 5: Establishing dummy and Naive Bayes baselines using stratified train-test splits.)*

# 11. Initial Evaluation

*(Planned for Stage 6: Benchmarking performance metrics, Confusion Matrix, and PR-AUC.)*

# 12. Observations and Conclusions

### Stage 3B Summary (EDA & Text Profiling)
- **Stage 3B.1 Completed:** Verified 3,201 cleaned emails across Not Spam (84.97%) and Spam (15.03%), established length/word count distributions, and generated 5 baseline graphs.
- **Stage 3B.2 Completed:** Extracted class-independent n-grams (unigrams, bigrams, trigrams), profiled 6 structural heuristic metrics (URLs, exclamation marks, question marks, currency symbols, uppercase ratio, all-caps words), and generated 3 comparative figures.
- Confirmed that punctuation, casing, and currency markers provide strong discriminating signals, establishing a clear academic rationale for feature preservation during Stage 4 Scikit-learn Pipeline construction.
- Ready to proceed to text preprocessing and pipeline engineering upon approval.